# Scraping/Downloading data from Various Sources

## Update on Scraping and Downloading data:
### 3/2/2026


#### Unusable Data Sources:
- ***Fbref has now changed, and scraping data from them is no longer possible***
- Tried multiple different sources, and corroborated with sources online
    - https://github.com/probberechts/soccerdata/issues/916
- instead switched to Understat for general and player specific information

#### Proposed Process For End User Data Extraction Process
1. Download all data for players performances, and store it in a csv(or in a database)
2. Query the requested data from the user as required

In [22]:
import soccerdata as sd

sd.__version__

'1.8.8'

In [ ]:
import soccerdata as sd
import pandas as pd
from pathlib import Path

OUT = Path("data/raw_csv")
OUT.mkdir(parents=True, exist_ok=True)

league = "ENG-Premier League"
season = 2024

# FotMob
fm = sd.FotMob(leagues=league, seasons=season)
fm_sched = fm.read_schedule()
print("FotMob schedule:", fm_sched.shape)
display(fm_sched.head())
fm_sched.to_csv(OUT / f"fotmob__{league}__{season}__schedule.csv", index=False)

# Understat
us = sd.Understat(leagues=league, seasons=season)
us_player = us.read_player_season_stats()
print("Understat player season:", us_player.shape)
display(us_player.head())
us_player.to_csv(OUT / f"understat__{league}__{season}__player_season.csv", index=False)

us_team_match = us.read_team_match_stats()
print("Understat team match:", us_team_match.shape)
display(us_team_match.head())
us_team_match.to_csv(OUT / f"understat__{league}__{season}__team_match.csv", index=False)

In [ ]:
fm_match_stats = fm.read_team_match_stats()
print("FotMob match_stats:", fm_match_stats.shape)
display(fm_match_stats.head())
fm_match_stats.to_csv(OUT / f"fotmob__{league}__{season}__match_stats.csv", index=False)

In [1]:
import requests
import json
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access the API key using os.environ or os.getenv
api_key = os.getenv("FBO_APIKEY")

uri = 'https://api.football-data.org/v4/matches'
headers = { 'X-Auth-Token': api_key }

response = requests.get(uri, headers=headers)
for match in response.json()['matches']:
  print(match)

{'area': {'id': 2077, 'name': 'Europe', 'code': 'EUR', 'flag': 'https://crests.football-data.org/EUR.svg'}, 'competition': {'id': 2001, 'name': 'UEFA Champions League', 'code': 'CL', 'type': 'CUP', 'emblem': 'https://crests.football-data.org/CL.png'}, 'season': {'id': 2454, 'startDate': '2025-09-16', 'endDate': '2026-05-30', 'currentMatchday': 7, 'winner': None}, 'id': 552001, 'utcDate': '2026-01-21T17:45:00Z', 'status': 'TIMED', 'matchday': 7, 'stage': 'LEAGUE_STAGE', 'group': None, 'lastUpdated': '2026-01-21T01:32:00Z', 'homeTeam': {'id': 611, 'name': 'Qarabağ Ağdam FK', 'shortName': 'Qarabağ Ağdam', 'tla': 'QAR', 'crest': 'https://crests.football-data.org/611.png'}, 'awayTeam': {'id': 19, 'name': 'Eintracht Frankfurt', 'shortName': 'Frankfurt', 'tla': 'SGE', 'crest': 'https://crests.football-data.org/19.png'}, 'score': {'winner': None, 'duration': 'REGULAR', 'fullTime': {'home': None, 'away': None}, 'halfTime': {'home': None, 'away': None}}, 'odds': {'msg': 'Activate Odds-Package in

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import date
from pathlib import Path
from typing import Iterable, Literal

import pandas as pd
import soccerdata as sd


# -------------------------
# Config / Helpers
# -------------------------

Top5 = Literal["EPL", "La Liga", "Bundesliga", "Serie A", "Ligue 1"]

TOP5_UNDERSTAT: list[Top5] = ["EPL", "La Liga", "Bundesliga", "Serie A", "Ligue 1"]
TOP5_FOTMOB_HINTS: dict[Top5, list[str]] = {
    "EPL": ["ENG-Premier League", "Premier League"],
    "La Liga": ["ESP-LaLiga", "LaLiga", "La Liga"],
    "Bundesliga": ["GER-Bundesliga", "Bundesliga"],
    "Serie A": ["ITA-Serie A", "Serie A"],
    "Ligue 1": ["FRA-Ligue 1", "Ligue 1"],
}


def last_n_seasons(
    n: int = 10,
    *,
    end_season_start_year: int | None = None,
    include_current_season: bool = False,
) -> list[str]:
    """
    Return seasons as strings like '2015-16', '2016-17', ...
    - If include_current_season=False, ends at the most recently *completed* season.
    - If include_current_season=True, includes the current in-progress season.
    """
    today = date.today()

    # European season typically starts in Aug.
    # If we're before Aug, the current season started last year.
    current_start_year = today.year if today.month >= 8 else today.year - 1

    if end_season_start_year is None:
        end_season_start_year = current_start_year if include_current_season else current_start_year - 1

    start_year = end_season_start_year - (n - 1)

    seasons = []
    for y in range(start_year, end_season_start_year + 1):
        seasons.append(f"{y}-{str((y + 1) % 100).zfill(2)}")  # 2015-16 style
    return seasons


def _safe_mkdir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def _save_csv(df: pd.DataFrame, outpath: Path) -> None:
    _safe_mkdir(outpath.parent)
    df.to_csv(outpath, index=False)


def resolve_fotmob_top5_leagues() -> list[str]:
    """
    FotMob league IDs can vary in exact naming.
    This function:
      1) pulls available FotMob league IDs,
      2) selects best matches for the Top 5 using simple string hints,
      3) falls back to keyword contains.
    """
    available = sd.FotMob.available_leagues()  # list[str] :contentReference[oaicite:3]{index=3}
    chosen: list[str] = []

    def pick_one(hints: list[str]) -> str | None:
        # Exact / preferred match first
        for h in hints:
            if h in available:
                return h
        # Substring fallback
        lower_avail = {a.lower(): a for a in available}
        for h in hints:
            h_low = h.lower()
            for a_low, a in lower_avail.items():
                if h_low in a_low:
                    return a
        return None

    for league in TOP5_UNDERSTAT:
        pick = pick_one(TOP5_FOTMOB_HINTS[league])
        if pick is not None:
            chosen.append(pick)

    # If your environment returns different league IDs, print available and adjust hints.
    if len(chosen) < 5:
        missing = [l for l in TOP5_UNDERSTAT if not any(h in " ".join(chosen) for h in TOP5_FOTMOB_HINTS[l])]
        raise ValueError(
            "Could not resolve all Top 5 leagues for FotMob.\n"
            f"Resolved: {chosen}\nMissing (hints may need updating): {missing}\n"
            "Tip: print(sd.FotMob.available_leagues()) and update TOP5_FOTMOB_HINTS."
        )

    return chosen


@dataclass(frozen=True)
class FetchResult:
    understat_player_season: pd.DataFrame
    understat_team_match: pd.DataFrame
    understat_schedule: pd.DataFrame
    fotmob_schedule: pd.DataFrame
    fotmob_team_match_stats: pd.DataFrame


# -------------------------
# Understat Fetchers
# -------------------------

def fetch_understat_player_season_stats(
    leagues: Iterable[str],
    seasons: Iterable[str],
    *,
    force_cache: bool = False,
) -> pd.DataFrame:
    us = sd.Understat(leagues=list(leagues), seasons=list(seasons))
    return us.read_player_season_stats(force_cache=force_cache)  # :contentReference[oaicite:4]{index=4}


def fetch_understat_team_match_stats(
    leagues: Iterable[str],
    seasons: Iterable[str],
    *,
    force_cache: bool = False,
) -> pd.DataFrame:
    us = sd.Understat(leagues=list(leagues), seasons=list(seasons))
    return us.read_team_match_stats(force_cache=force_cache)  # :contentReference[oaicite:5]{index=5}


def fetch_understat_schedule(
    leagues: Iterable[str],
    seasons: Iterable[str],
    *,
    include_matches_without_data: bool = True,
    force_cache: bool = False,
) -> pd.DataFrame:
    us = sd.Understat(leagues=list(leagues), seasons=list(seasons))
    return us.read_schedule(
        include_matches_without_data=include_matches_without_data,
        force_cache=force_cache,
    )  # :contentReference[oaicite:6]{index=6}


# -------------------------
# FotMob Fetchers
# -------------------------

def fetch_fotmob_schedule(
    leagues: Iterable[str],
    seasons: Iterable[str],
    *,
    force_cache: bool = False,
) -> pd.DataFrame:
    fm = sd.FotMob(leagues=list(leagues), seasons=list(seasons))
    return fm.read_schedule(force_cache=force_cache)  # :contentReference[oaicite:7]{index=7}


def fetch_fotmob_team_match_stats(
    leagues: Iterable[str],
    seasons: Iterable[str],
    *,
    stat_type: str = "Top stats",
    opponent_stats: bool = True,
    team: str | list[str] | None = None,
    force_cache: bool = False,
) -> pd.DataFrame:
    fm = sd.FotMob(leagues=list(leagues), seasons=list(seasons))
    return fm.read_team_match_stats(
        stat_type=stat_type,
        opponent_stats=opponent_stats,
        team=team,
        force_cache=force_cache,
    )  # :contentReference[oaicite:8]{index=8}


# -------------------------
# End-to-end: Top 5 x last 10 seasons
# -------------------------

def fetch_top5_last10(
    *,
    seasons: list[str] | None = None,
    include_current_season: bool = False,
    fotmob_leagues: list[str] | None = None,
    export_dir: str | Path | None = None,
    force_cache: bool = False,
    fotmob_stat_type: str = "Top stats",
) -> FetchResult:
    """
    Pulls:
      - Understat player season stats
      - Understat team match stats + schedule
      - FotMob schedule + team match stats

    If export_dir is provided, writes CSVs under that folder.
    """
    if seasons is None:
        seasons = last_n_seasons(10, include_current_season=include_current_season)

    understat_leagues = TOP5_UNDERSTAT
    if fotmob_leagues is None:
        fotmob_leagues = resolve_fotmob_top5_leagues()

    us_player = fetch_understat_player_season_stats(understat_leagues, seasons, force_cache=force_cache)
    us_team_match = fetch_understat_team_match_stats(understat_leagues, seasons, force_cache=force_cache)
    us_sched = fetch_understat_schedule(understat_leagues, seasons, force_cache=force_cache)

    fm_sched = fetch_fotmob_schedule(fotmob_leagues, seasons, force_cache=force_cache)
    fm_team_match = fetch_fotmob_team_match_stats(
        fotmob_leagues,
        seasons,
        stat_type=fotmob_stat_type,
        opponent_stats=True,
        force_cache=force_cache,
    )

    if export_dir is not None:
        export_dir = Path(export_dir)
        _save_csv(us_player, export_dir / "understat__top5__player_season.csv")
        _save_csv(us_team_match, export_dir / "understat__top5__team_match.csv")
        _save_csv(us_sched, export_dir / "understat__top5__schedule.csv")
        _save_csv(fm_sched, export_dir / "fotmob__top5__schedule.csv")
        _save_csv(fm_team_match, export_dir / f"fotmob__top5__team_match_stats__{fotmob_stat_type.replace(' ', '_')}.csv")

    return FetchResult(
        understat_player_season=us_player,
        understat_team_match=us_team_match,
        understat_schedule=us_sched,
        fotmob_schedule=fm_sched,
        fotmob_team_match_stats=fm_team_match,
    )

[02/23/26 00:19:28] INFO     No custom team name replacements found. You can configure these in       ]8;id=746864;file:///home/user/projects/adv-soccer-stats/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=668759;file:///home/user/projects/adv-soccer-stats/.venv/lib/python3.12/site-packages/soccerdata/_config.py#92\92]8;;\
                             /home/user/soccerdata/config/teamname_replacements.json.                              

                    INFO     No custom league dict found. You can configure additional leagues in    ]8;id=448939;file:///home/user/projects/adv-soccer-stats/.venv/lib/python3.12/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=24425;file:///home/user/projects/adv-soccer-stats/.venv/lib/python3.12/site-packages/soccerdata/_config.py#198\198]8;;\
                             /home/user/soccerdata/config/league_dict.json.                                        

In [2]:
if __name__ == "__main__":
    seasons = last_n_seasons(10, include_current_season=False)
    print("Seasons:", seasons)

    out = fetch_top5_last10(
        seasons=seasons,
        export_dir="data/raw_csv",
        force_cache=False,
        fotmob_stat_type="Top stats",
    )

    print(out.understat_player_season.shape, out.fotmob_team_match_stats.shape)

Seasons: ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


ValueError: 
                        Invalid league 'EPL'. Valid leagues are:
                        ['ENG-Premier League',
 'ESP-La Liga',
 'FRA-Ligue 1',
 'GER-Bundesliga',
 'ITA-Serie A']
                        